In [107]:
from InputData import *

# Reduced
#instance_filename = "Construction_a1_o12_m3_an5_ar3_reduced.json"
#instance_filename = "Construction_a3_o80_m10_an10_ar9_reduced.json"
#instance_filename = "Construction_a5_o96_m10_an10_ar10_reduced.json"

# 10 Sites --> "Construction_a10_o118_m6_an53_ar13.json": Instance not duable since one order has no order items
instance_filename = "Construction_a10_o107_m5_an57_ar12.json"
#instance_filename = "Construction_a10_o114_m6_an57_ar11.json"
#instance_filename = "Construction_a10_o119_m5_an54_ar13.json"
#instance_filename = "Construction_a10_o144_m6_an53_ar12.json"

# 15 Sites
#instance_filename = "Construction_a15_o191_m8_an74_ar18.json"

# 20 Sites
#instance_filename = "Construction_a20_o259_m11_an101_ar26.json"

# 50 Sites
#instance_filename = "Construction_a50_o578_m28_an276_ar66.json"

data = InputData(instance_filename)


Data loaded from 'Construction_a10_o107_m5_an57_ar12.json' in folder 'AnzahlAuftraege_NEW_10'.


In [108]:
import gurobipy as gp
from gurobipy import GRB
import json
from pathlib import Path
import pandas as pd
from InputData import *
from OutputData import *
from time import time
from itertools import groupby


class FlowFormulation:
    
    def __init__(self, data):
        self.data = data
        self.model = None
        #self.objective_strategy = objective_strategy

        # ========================
        # 1. Sets
        # ========================
        self.M = []  # List of machine IDs
        self.W_m = {}  # Dictionary: Regular drivers for machines
        self.N_m = {}  # Dictionary: Machine -> Order items
        self.W = []  # List of worker IDs
        self.N_w = {}  # Dictionary: Worker -> Order items
        self.N = []  # List of order item IDs
        self.C = []  # List of site IDs
        self.N_c = {}  # Dictionary: Site -> Order items

        # ========================
        # 2. Parameters
        # ========================
        self.start_date = None  # Start date of the planning horizon
        self.end_date = None # End date of the planning horizon
        self.O_t = {}  # Dictionary: Day -> Order items (start)
        self.D_r = {}  # Dictionary: Day -> Day shifts
        self.N_r = {}  # Dictionary: Day -> Night shifts
        self.A_r = {}  # Dictionary: Day -> All shifts
        self.O_t_start = {}  # Dictionary: Start times of order items
        self.O_t_end = {}  # Dictionary: End times of order items
        self.O_t_start_inverted = {}  # Dictionary: Order item -> Start time
        self.O_t_end_inverted = {}  # Dictionary: Order item -> End time

        # ========================
        # 3. Predecessors, Successors and Distances
        # ========================
        self.P_mn = {}  # Predecessors for machine order items
        self.S_mn = {}  # Successors for machine order items
        self.P_wn = {}  # Predecessors for worker order items
        self.S_wn = {}  # Successors for worker order items
        self.d_ij = []  # Distance matrix for machines (transport routes)
        self.d_wi = []  # Distance matrix for workers (work routes)

        # ========================
        # 4. Time and Range
        # ========================
        self.T_range = []  # List of all days in the planning horizon
        self.T = 0  # Planning horizon (number of days)
        self.start = "start" # Start node which is indexed as len(N)
        self.end = "end" # End node which is indexed as len(N) + 1

        # ========================
        # 5. Occupational Safety Constants
        # ========================
        self.S_Nmax = data._consecutive_night_shifts
        self.S_max = data._max_shifts_in_time_period
        self.T_Smax = data._time_period_for_max_shifts
        self.T_Wmax = data._max_working_hours

        # ========================
        # 6. Other Constants
        # ========================
        self.SECONDS_IN_A_DAY = data._seconds_a_day
        self.TRANSPORT_SPEED = data._transport_speed_kmh * 24  # Machine transport speed (km/day)
        self.TIME_BETWEEN_SHIFTS = data._hours_between_shifts / 24  # Rest period between shifts (in days)

        
    def preprocess_data(self):
        """Preprocess the input data for optimization."""
        print("\nPreprocessing data...")
        current_time = time()
        
        # ========================
        # 1. Process Machines
        # ========================
        for machine in self.data.machines:
            self.M.append(machine.name)
            self.W_m[machine.name] = [int(driver) for driver in machine.default_drivers]
            self.N_m[machine.name] = []
            for orderItem in self.data.order_items:
                if orderItem.machine_type == machine.type:
                    self.N_m[machine.name].append(orderItem.id)

        # ========================
        # 2. Process Workers
        # ========================
        for worker in self.data.workers:
            self.W.append(worker.personal_number)
            self.N_w[worker.personal_number] = []
            for orderItem in self.data.order_items:
                if not orderItem.worker_qualifications:  # Keine Qualifikationen erforderlich
                    self.N_w[worker.personal_number].append(orderItem.id)
                elif set(orderItem.worker_qualifications).issubset(set(worker.qualifications)):  # Qualifikationen sind abgedeckt
                    self.N_w[worker.personal_number].append(orderItem.id)

        # ========================
        # 3. Process Orders
        # ========================
        for order in self.data.orders:
            self.C.append(order.site_number)
            self.N_c[order.site_number] = [int(item_id) for item_id in order.order_item_ids]

        # ========================
        # 4. Process Order Items
        # ========================
        self.N = [orderItem.id for orderItem in self.data.order_items]
        self.start_date = self.data.start_date
        self.end_date = self.data.end_date

        for orderItem in self.data.order_items:
            orderID = orderItem.id
            delta_start = (orderItem.start_time - self.start_date)
            t_start = delta_start.total_seconds() / self.SECONDS_IN_A_DAY
            t_start_int = int(t_start)

            # O_t: Order items grouped by day
            if t_start_int not in self.O_t:
                self.O_t[t_start_int] = []
            self.O_t[t_start_int].append(orderID)

            # D_r: Day shifts grouped by day
            if t_start_int not in self.D_r:
                self.D_r[t_start_int] = []
            if orderItem.start_time.hour <= self.data._day_and_night_shift_boundary:
                self.D_r[t_start_int].append(orderID)

            # N_r: Night shifts grouped by day
            if t_start_int not in self.N_r:
                self.N_r[t_start_int] = []
            if orderItem.start_time.hour > self.data._day_and_night_shift_boundary:
                self.N_r[t_start_int].append(orderID)

            # A_r: All shifts grouped by day
            if t_start_int not in self.A_r:
                self.A_r[t_start_int] = []
            self.A_r[t_start_int].append(orderID)

            # Start times
            if t_start not in self.O_t_start:
                self.O_t_start[t_start] = []
            self.O_t_start[t_start].append(orderID)
            self.O_t_start_inverted[orderID] = t_start

            # End times
            delta_end = (orderItem.end_time - self.start_date)
            t_end = delta_end.total_seconds() / self.SECONDS_IN_A_DAY
            if t_end not in self.O_t_end:
                self.O_t_end[t_end] = []
            self.O_t_end[t_end].append(orderID)
            self.O_t_end_inverted[orderID] = t_end

        # ========================
        # 5. Process Transport Routes
        # ========================
        for i in self.data.order_items:
            row = []
            for j in self.data.order_items:
                a = next((k for k, v in self.N_c.items() if i.id in v))
                b = next((k for k, v in self.N_c.items() if j.id in v))
                row.append(self.data.transport_routes[a][b])
            self.d_ij.append(row)

        for i in self.data.workers:
            row = []
            for j in self.data.order_items:
                a = next((k for k, v in self.N_c.items() if j.id in v))
                row.append(self.data.work_routes[i.personal_number][a])
            self.d_wi.append(row)

        # ========================
        # 6. Calculate Predecessors and Successors
        # ========================

        for m in self.M:
            for n in self.N_m[m]:
                if (m, self.start) not in self.P_mn:
                    self.P_mn[m, self.start] = []
                    self.S_mn[m, self.start] = [self.end]
                if (m, self.end) not in self.P_mn:
                    self.P_mn[m, self.end] = []
                    self.S_mn[m, self.end] = [self.start]

                self.P_mn[m, n] = [self.start]
                self.S_mn[m, self.start].append(n)
                self.P_mn[m, self.end].append(n)
                self.S_mn[m, n] = [self.end]

                for i in self.N_m[m]:
                    if n != i:
                        start_time_n = self.O_t_start_inverted[n]
                        end_time_n = self.O_t_end_inverted[n]
                        start_time_i = self.O_t_start_inverted[i]
                        end_time_i = self.O_t_end_inverted[i]

                        if start_time_n >= end_time_i + self.d_ij[i][n] / self.TRANSPORT_SPEED:
                            self.P_mn[m, n].append(i)

                        if start_time_i > end_time_n + self.d_ij[n][i] / self.TRANSPORT_SPEED:
                            self.S_mn[m, n].append(i)

        for w in self.W:
            for n in self.N_w[w]:
                if (w, self.start) not in self.P_wn:
                    self.P_wn[w, self.start] = []
                    self.S_wn[w, self.start] = [self.end]
                if (w, self.end) not in self.P_wn:
                    self.P_wn[w, self.end] = []
                    self.S_wn[w, self.end] = [self.start]

                self.P_wn[w, n] = [self.start]
                self.S_wn[w, self.start].append(n)
                self.P_wn[w, self.end].append(n)
                self.S_wn[w, n] = [self.end]

                for i in self.N_w[w]:
                    if n != i:
                        start_time_n = self.O_t_start_inverted[n]
                        end_time_n = self.O_t_end_inverted[n]
                        start_time_i = self.O_t_start_inverted[i]
                        end_time_i = self.O_t_end_inverted[i]

                        if start_time_n >= end_time_i + self.TIME_BETWEEN_SHIFTS:
                            self.P_wn[w, n].append(i)

                        if start_time_i >= end_time_n + self.TIME_BETWEEN_SHIFTS:
                            self.S_wn[w, n].append(i)

        # ========================
        # 7. Time Range and Planning Horizon
        # ========================
        day_difference = self.end_date - self.start_date
        self.T_range = list(range(day_difference.days + 1))
        self.T = day_difference.days + 1
        
        '''
        end_date_adjusted = self.start_date
        for orderItem in self.data.order_items:
            if end_date_adjusted < orderItem.start_time:
                end_date_adjusted = orderItem.start_time
        self.T = (end_date_adjusted - self.start_date).days + 1
        '''

        # ========================
        # 8. Order Item Durations
        # ========================
        self.t_o = [orderItem.duration for orderItem in self.data.order_items]

        elapsed_time = time() - current_time
        print("Data preprocessed successfully.")
        print(f"Time elapsed: {elapsed_time:.2f} seconds")



    def create_optimization_model(self):
        """Create and configure the Gurobi optimization model."""

        current_time = time()
        print("\nCreating optimization model...")
        self.model = gp.Model("Flow_Formulation")

        # ========================
        # 1. Create Variables
        # ========================
        # Machine flow variables
        indices_1 = [(m, i, j) for m in self.M for i in self.N_m[m] for j in self.N_m[m]]  # (m, i, j)
        indices_2 = [(m, self.start, j) for m in self.M for j in self.N_m[m]]  # (m, start, j)
        indices_3 = [(m, i, self.end) for m in self.M for i in self.N_m[m]]  # (m, i, end)
        indices_4 = [(m, self.start, self.end) for m in self.M]  # (m, start, end)
        all_indices = indices_1 + indices_2 + indices_3 + indices_4
        x = self.model.addVars(all_indices, vtype=GRB.BINARY, name="x")

        # Worker flow variables
        indices_1 = [(w, i, j) for w in self.W for i in self.N_w[w] for j in self.N_w[w]]  # (w, i, j)
        indices_2 = [(w, self.start, j) for w in self.W for j in self.N_w[w]]  # (w, start, j)
        indices_3 = [(w, i, self.end) for w in self.W for i in self.N_w[w]]  # (w, i, end)
        indices_4 = [(w, self.start, self.end) for w in self.W]  # (w, start, end)
        all_indices = indices_1 + indices_2 + indices_3 + indices_4
        y = self.model.addVars(all_indices, vtype=GRB.BINARY, name="y")

        # Non-regular driver utilization variables
        r = self.model.addVars(self.N, vtype=GRB.BINARY, name="r")

        # Site completion variables
        u = self.model.addVars(self.C, vtype=GRB.BINARY, name="u")

        # ========================
        # 2. Set Objective Function
        # ========================

        self.objective_strategy = "weighted"


        # Definition of the objective criteria/functions
        self.construction_fulfillment = gp.quicksum(u[c] for c in self.C)
        self.machine_transport_distance = gp.quicksum(self.d_ij[i][j] * x[m, i, j] for m in self.M for i in self.N_m[m] for j in self.N_m[m])
        self.worker_work_distance = gp.quicksum(2 * self.d_wi[w][i] * y[w, i, j] for w in self.W for i in self.N_w[w] for j in (self.N_w[w] + [self.end]))
        self.machine_usage = gp.quicksum(x[m, self.start, j] for m in self.M for j in self.N_m[m])
        self.worker_usage = gp.quicksum(y[w, self.start, j] for w in self.W for j in self.N_w[w])
        self.non_regular_driver_usage = gp.quicksum(r[i] for i in self.N)


        if self.objective_strategy == "single":
            
            self.model.setObjectiveN(-self.construction_fulfillment, index=0, weight = self.data._construction_revenue)
            
            self.model.setObjectiveN(self.machine_transport_distance, index=1, weight = self.data._machine_transport_cost_per_km)
            self.model.setObjectiveN(self.worker_work_distance, index=2, weight = self.data._worker_travel_cost_per_km)
            self.model.setObjectiveN(self.machine_usage, index=3, weight = self.data._machine_fixed_cost)
            self.model.setObjectiveN(self.worker_usage, index=4, weight = self.data._worker_fixed_cost)
            self.model.setObjectiveN(self.non_regular_driver_usage, index=5, weight = self.data._penalty_cost_non_regular_driver)
            


        elif self.objective_strategy == "weighted":
            
            # Predefining elements for the weights
            len_unique_machine_types = list()
            
            for order in self.data.orders:
                machine_types = []
                for orderItemID in order.order_item_ids:
                    orderItemID = int(orderItemID)
                    orderItem = next((orderItem for orderItem in self.data.order_items if orderItem.id == orderItemID))

                    machine_types.append(orderItem.machine_type)

                unique_machine_types = list(set(machine_types))
                len_unique_machine_types.append(len(unique_machine_types))

            average_order_duration = sum(item.duration for item in self.data.order_items) / len(self.C)
            average_machine_types_per_site = sum(len_unique_machine_types) / len(self.C)
            average_transport_distance = sum(item for row in self.data.transport_routes for item in row if item != 0) / sum(1 for row in self.data.transport_routes for item in row if item != 0) 
            average_order_items_per_site = len(self.N) / len(self.C)
            target_max_share_of_non_regular_drivers = 0.3
            average_work_distance = sum(item for row in self.d_wi for item in row if item != 0) / sum(1 for row in self.d_wi for item in row if item != 0)
            


            # Calculating the weights
            self.non_regular_driver_usage_weight = average_order_items_per_site * target_max_share_of_non_regular_drivers
            self.transport_distance_weight = average_machine_types_per_site * 2 * average_transport_distance
            self.work_distance_weight = average_order_items_per_site * 2 * average_work_distance
            self.machine_usage_weight = average_machine_types_per_site
            self.worker_usage_weight = (average_order_duration / self.T_Wmax)



            # Setting the objective function
            self.model.setObjectiveN(-self.construction_fulfillment, index=0, weight = 1)
            
            self.model.setObjectiveN(self.machine_transport_distance, index=1, weight = self.transport_distance_weight)
            self.model.setObjectiveN(self.worker_work_distance, index=2, weight = self.work_distance_weight)
            self.model.setObjectiveN(self.machine_usage, index=3, weight = self.machine_usage_weight)
            self.model.setObjectiveN(self.worker_usage, index=4, weight = self.worker_usage_weight)
            self.model.setObjectiveN(self.non_regular_driver_usage, index=5, weight = self.non_regular_driver_usage_weight)



        elif self.objective_strategy == "hierarchical":

            self.model.setObjectiveN(-self.construction_fulfillment, index=0, priority=6)
            
            self.model.setObjectiveN(self.machine_transport_distance, index=1, priority=3)
            self.model.setObjectiveN(self.worker_work_distance, index=2, priority=3)
            
            self.model.setObjectiveN(self.machine_usage, index=3, priority=5)
            self.model.setObjectiveN(self.worker_usage, index=4, priority=5)
            
            self.model.setObjectiveN(self.non_regular_driver_usage, index=5, priority=1)




        elif self.objective_strategy == "epsilon_constraint":

            # Main objective function: Construction fulfillment
            self.model.setObjective(self.construction_fulfillment,GRB.MAXIMIZE)

            # ε-Values
            self.epsilon_machine_use = round(len(self.M) * 0.7)
            self.epsilon_worker_use = round(len(self.W) * 0.7)
            
            self.epsilon_machine_distance = round((len(self.C)/self.epsilon_machine_use) * 500 * 0.7)
            self.epsilon_worker_distance = round((len(self.N)/self.epsilon_worker_use) * 200 * 0.7)
            
            self.epsilon_non_regular_driver_use = round(len(self.N) * 0.2)

            # ε-Constraints
            self.model.addConstr(self.machine_transport_distance <= self.epsilon_machine_distance,name= "EpsilonMachineDistanceConstraint")
            self.model.addConstr(self.worker_work_distance <= self.epsilon_worker_distance, name="EpsilonWorkerDistanceConstraint")
            self.model.addConstr(self.machine_usage <= self.epsilon_machine_use, name="EpsilonMachineUsageConstraint")
            self.model.addConstr(self.worker_usage <= self.epsilon_worker_use, name="EpsilonWorkerUsageConstraint")
            self.model.addConstr(self.non_regular_driver_usage <= self.epsilon_non_regular_driver_use, name="EpsilonPenaltyCostConstraint")


        # ========================
        # 3. Add Constraints
        # ========================
        # Machine flow balance constraints
        for m in self.M:
            for i in self.N_m[m]:
                self.model.addConstr(
                    gp.quicksum(x[m, j, i] for j in self.P_mn[m, i]) ==
                    gp.quicksum(x[m, i, j] for j in self.S_mn[m, i]),
                    name=f"machine_flow_balance_{m}_{i}"
                )

        # Worker flow balance constraints
        for w in self.W:
            for i in self.N_w[w]:
                self.model.addConstr(
                    gp.quicksum(y[w, j, i] for j in self.P_wn[w, i]) ==
                    gp.quicksum(y[w, i, j] for j in self.S_wn[w, i]),
                    name=f"worker_flow_balance_{w}_{i}"
                )

        # Start and end node constraints for machines
        for m in self.M:
            if (m, self.start) in self.S_mn:
                self.model.addConstr(
                    gp.quicksum(x[m, self.start, j] for j in self.S_mn[m, self.start]) == 1,
                    name=f"machine_start_constraint_{m}"
                )
        # Start and end node constraints for workers
        for w in self.W:
            if (w, self.start) in self.S_wn:
                self.model.addConstr(
                    gp.quicksum(y[w, self.start, j] for j in self.S_wn[w, self.start]) == 1,
                    name=f"worker_start_constraint_{w}"
                )

        # Regular driver constraints
        for m in self.M:
            for i in self.N_m[m]:
                self.model.addConstr(
                    gp.quicksum(x[m, i, j] for j in self.S_mn[m, i]) <=
                    gp.quicksum(y[w, i, j] for w in self.W_m[m] if (w, i) in self.S_wn for j in self.S_wn[w, i]) + r[i],
                    name=f"regular_driver_constraint_{m}_{i}"
                )

        # Site completion constraints
        for c in self.C:
            for i in self.N_c[c]:
                self.model.addConstr(
                    gp.quicksum(x[m, i, j] for m in self.M if (m, i) in self.S_mn for j in self.S_mn[m, i]) == u[c],
                    name=f"machine_site_fulfillment_site{c}_order{i}"
                )
                self.model.addConstr(
                    gp.quicksum(y[w, i, j] for w in self.W if (w, i) in self.S_wn for j in self.S_wn[w, i]) == u[c],
                    name=f"worker_site_fulfillment_site{c}_order{i}"
                )

        # Night shift constraints
        for w in self.W:
            for t in self.T_range:
                if t <= self.T - self.S_Nmax:
                    self.model.addConstr(
                        gp.quicksum(
                            y[w, i, j] for t_ in range(t, t + self.S_Nmax + 1) if t_ in self.N_r for j in self.N_r[t_]
                            if (w, j) in self.P_wn for i in self.P_wn[w, j]
                        ) <= self.S_Nmax,
                        name=f"night_shift_constraint_{w}_t{t}"
                    )

        # Shift count constraints
        for w in self.W:
            for t in self.T_range:
                if t <= self.T - self.T_Smax:
                    self.model.addConstr(
                        gp.quicksum(
                            y[w, i, j] for t_ in range(t, t + self.T_Smax) if t_ in self.A_r for j in self.A_r[t_]
                            if (w, j) in self.P_wn for i in self.P_wn[w, j]
                        ) <= self.S_max,
                        name=f"shift_number_constraint_{w}_t{t}"
                    )

        # Total working time constraints
        for w in self.W:
            self.model.addConstr(
                gp.quicksum(self.t_o[i] * y[w, i, j] for i in self.N_w[w] for j in self.S_wn[w, i]) <= self.T_Wmax,
                name=f"work_time_constraint_{w}"
            )
        
        elapsed_time = time() - current_time
        print("Optimization model created successfully.")
        print(f"Time elapsed: {elapsed_time:.2f} seconds")




    def execute(self):
        """Run the full optimization workflow."""
        
        self.preprocess_data()
        self.create_optimization_model()

        return self.worker_usage_weight


# Execute the optimization workflow
flow_formulation = FlowFormulation(data)
average_transport_distance = flow_formulation.execute()
print(average_transport_distance)



Preprocessing data...
Data preprocessed successfully.
Time elapsed: 0.07 seconds

Creating optimization model...
Optimization model created successfully.
Time elapsed: 0.69 seconds
0.7106250000000001


In [109]:
matrix = flow_formulation.d_wi


sum(item for row in matrix for item in row if item != 0) / sum(1 for row in matrix for item in row if item != 0)

54.40574325918168